In [2]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

class BertTextEncoder(nn.Module):
    def __init__(self, model_name="cl-tohoku/bert-base-japanese-whole-word-masking", 
                 output_embedding_dim=None, freeze_bert=True):
        """
        BERTを用いたテキストエンコーダ。

        Args:
            model_name (str): 使用するHugging Faceの事前学習済みモデル名。
            output_embedding_dim (int, optional): 
                BERTの出力特徴量をこの次元に射影するための全結合層の出力次元。
                Noneの場合、BERTの生の出力特徴量（通常768次元）をそのまま利用します。
            freeze_bert (bool): Trueの場合、BERTの重みを凍結してファインチューニングしません。
        """
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.bert_model = AutoModel.from_pretrained(model_name)
        
        self.bert_output_dim = self.bert_model.config.hidden_size # 通常は768

        if freeze_bert:
            for param in self.bert_model.parameters():
                param.requires_grad = False
        
        # オプション: BERTの出力をさらに指定次元に射影する全結合層
        if output_embedding_dim and output_embedding_dim != self.bert_output_dim:
            self.projection_layer = nn.Linear(self.bert_output_dim, output_embedding_dim)
        else:
            self.projection_layer = nn.Identity() # 射影しない場合はそのまま出力

    def forward(self, texts):
        """
        テキストのリストを受け取り、埋め込みベクトルのバッチを返します。

        Args:
            texts (list of str): エンコードするテキストのリスト (バッチ)。例: ["コメント1", "コメント2", ...]

        Returns:
            torch.Tensor: テキスト埋め込みのバッチ。形状は (batch_size, embedding_dim)。
                         embedding_dim は output_embedding_dim を指定した場合はその値、
                         指定しない場合はBERTの出力次元 (例: 768)。
        """
        # トークナイズ処理
        # padding=True: バッチ内で最長のシーケンスに合わせてパディング
        # truncation=True: モデルの最大長を超える場合は切り捨て
        # return_tensors='pt': PyTorchテンソルで結果を返す
        # device=self.bert_model.device: モデルと同じデバイスにテンソルを作成
        inputs = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            return_tensors='pt',
            max_length=self.tokenizer.model_max_length # モデルが扱える最大長 (通常512)
        ).to(self.bert_model.device)

        # BERTモデルによるエンコード
        # attention_maskも自動的に渡される
        outputs = self.bert_model(**inputs)

        # BERTの出力からテキスト全体の埋め込みベクトルを取得
        # 一般的には[CLS]トークンの隠れ状態 (outputs.last_hidden_state[:, 0, :]) を使用する
        # または、全トークンの隠れ状態の平均プーリング (outputs.last_hidden_state.mean(dim=1)) も考えられる
        # ここでは[CLS]トークン表現を使用
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # (batch_size, bert_output_dim)
        
        # オプションの射影層を適用
        projected_embedding = self.projection_layer(cls_embedding) # (batch_size, output_embedding_dim or bert_output_dim)
        
        return projected_embedding

In [4]:
# ---- 使用例 ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. BERTの生の出力をそのまま使う場合
text_encoder_raw = BertTextEncoder(
    model_name="cl-tohoku/bert-base-japanese-whole-word-masking",
    freeze_bert=True # 学習初期は凍結することが多い
).to(device)

# 2. BERTの出力を256次元に射影する場合 (対照学習で次元を揃えるためなど)
# この256という値は、局面エンコーダ側の出力次元や、対照学習で用いたい共通の埋め込み次元に合わせます。
projection_dim = 256 
text_encoder_projected = BertTextEncoder(
    model_name="cl-tohoku/bert-base-japanese-whole-word-masking",
    output_embedding_dim=projection_dim,
    freeze_bert=False # BERTもファインチューニングする場合
).to(device)

# ダミーの棋譜コメント
sample_comments = [
    "▲２六歩 △３四歩 ▲７六歩 △８四歩",
    "ここで▲２二角成と踏み込むのが好手でした。",
    "先手優勢。次の▲５五角が厳しい一手。",
    "難解な中盤戦が続いています。"
]

# エンコード実行 (生の出力)
text_encoder_raw.eval() # 評価モード
with torch.no_grad():
    embeddings_raw = text_encoder_raw(sample_comments)
print("BERT Raw Embeddings shape:", embeddings_raw.shape) # (batch_size, 768)

# エンコード実行 (射影後の出力)
text_encoder_projected.train() # 学習モード (Dropoutなどが有効になる場合)
# 実際には勾配計算が必要な場合は torch.no_grad() を外す
with torch.no_grad(): # ここでは例として勾配計算なし
    embeddings_projected = text_encoder_projected(sample_comments)
print("Projected Embeddings shape:", embeddings_projected.shape) # (batch_size, 256)

# --- トークナイザの動作確認 (参考) ---
tokenizer_check = AutoTokenizer.from_pretrained("cl-tohoku/bert-base-japanese-whole-word-masking")
sample_text = "▲２六歩 △３四歩"
tokenized_output = tokenizer_check(sample_text, padding=True, truncation=True, return_tensors='pt')
print("\nTokenized output for:", sample_text)
print("Input IDs:", tokenized_output['input_ids'])
print("Decoded tokens:", tokenizer_check.convert_ids_to_tokens(tokenized_output['input_ids'][0]))
print("Attention Mask:", tokenized_output['attention_mask'])

Some weights of the model checkpoint at cl-tohoku/bert-base-japanese-whole-word-masking were not used when initializing BertModel: ['cls.predictions.transform.dense.weight', 'cls.predictions.decoder.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of the model checkpoint at cl-tohoku/bert-base-japanese-whole-word-masking were not used when initializing BertM

BERT Raw Embeddings shape: torch.Size([4, 768])
Projected Embeddings shape: torch.Size([4, 256])

Tokenized output for: ▲２六歩 △３四歩
Input IDs: tensor([[    2, 21033,    25,  1688,  1513, 16196,    48,   755,  1513,     3]])
Decoded tokens: ['[CLS]', '▲', '2', '六', '歩', '△', '3', '四', '歩', '[SEP]']
Attention Mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [5]:
from dlshogi.network.policy_value_network import policy_value_network
from dlshogi import serializers
from dlshogi.common import MAX_MOVE_LABEL_NUM
class dlshogiEncoder(nn.Module):
    def __init__(self, dim_embedding: int, arg_network: str, arg_model: str):
        super().__init__()

        # dlshogiのpolicy networkとvalue networkの出力層前までのネットワークをバックボーンネットワークとする
        model = policy_value_network(arg_network)
        serializers.load_npz(arg_model, model)
        self.backbone = model

        # u21 の出力チャネル数 (k=192)
        self.dlshogi_feature_dim = 192 # k

        # GAP後の特徴量をさらに射影する層
        self.projection = nn.Linear(self.dlshogi_feature_dim, dim_embedding)

    '''
    エンコーダの順伝播
    features1: 入力1, [バッチサイズ, チャネル数(62), 高さ(9), 幅(9)]
    features2: 入力2, [バッチサイズ, チャネル数(57), 高さ(9), 幅(9)]
    '''
    def forward(self, features1: torch.Tensor, features2: torch.Tensor):
        with torch.no_grad():
            _, _, _, _, resnet_features = self.backbone(features1, features2)

        # Global Average Pooling
        pooled_features = F.adaptive_avg_pool2d(resnet_features, (1, 1)) # (batch, 192, 1, 1)
        board_features_raw = torch.flatten(pooled_features, 1) # (batch, 192)

        # 最終的な埋め込みベクトルに射影
        final_board_embedding = self.projection(board_features_raw) # (batch, projection_dim)
        
        return final_board_embedding

In [ ]:
# gemimi提案エンコーダ
class dlshogiEncoder(nn.Module):
    def __init__(self, dim_embedding: int, arg_network: str, arg_model: str, freeze_backbone: bool = True): # freeze_backboneフラグを追加
        super().__init__()
        model = policy_value_network(arg_network)
        serializers.load_npz(arg_model, model)
        self.backbone = model

        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        self.freeze_backbone_in_forward = freeze_backbone # forwardでも使うかどうかのフラグ

        self.dlshogi_feature_dim = 192
        self.projection = nn.Linear(self.dlshogi_feature_dim, dim_embedding)

    def forward(self, features1: torch.Tensor, features2: torch.Tensor):
        # self.freeze_backbone_in_forward が True の場合、または __init__ で requires_grad=False にした場合
        # ここでの torch.no_grad() は、主に requires_grad=True だが一時的に勾配計算を止めたい場合に効果的
        # __init__ で requires_grad=False に設定済みなら、ここでの no_grad() は必須ではないが、
        # 念のためや、他の部分で requires_grad が True に変更される可能性を考慮するなら残しても良い
        if self.freeze_backbone_in_forward: # もし __init__ で requires_grad=False にしないならこのif文は必須
            with torch.no_grad():
                # 5番目の戻り値がu21であると仮定
                outputs = self.backbone(features1, features2) 
                resnet_features = outputs[4] # インデックスでアクセスする方が安全か
        else:
            outputs = self.backbone(features1, features2)
            resnet_features = outputs[4]

        pooled_features = F.adaptive_avg_pool2d(resnet_features, (1, 1))
        board_features_raw = torch.flatten(pooled_features, 1)
        final_board_embedding = self.projection(board_features_raw)
        return final_board_embedding

In [ ]:
import os
import numpy as np
import logging
import torch
from torch.utils.data import Dataset, DataLoader
from cshogi import Board
from cshogi.dlshogi import make_input_features, FEATURES1_NUM, FEATURES2_NUM # これらの定数と関数が利用可能であると仮定します
import csv # CSVファイルを読み込むためにインポート

# --- ユーザー提供のdtype定義 (または実際の定義に合わせてください) ---
dtypeHcp = np.dtype((np.uint8, 32))
dtypeEval = np.dtype(np.int16)
dtypeMove16 = np.dtype(np.int16)
dtypeGameResult = np.dtype(np.int8)

HuffmanCodedPosAndEvalComment = np.dtype(
    [('hcp', dtypeHcp),
     ('eval', dtypeEval),
     ('bestMove16', dtypeMove16),
     ('gameResult', dtypeGameResult),
     ('dummy', np.uint8),
     ('comment_index', np.int32),
    ])
# -----------------------------------------------------------------

# ロガーの設定 (必要に応じて)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class ShogiClipDataset(Dataset):
    def __init__(self, hcpe_file_paths, comment_csv_file_path, preload_hcpe=True,
                 csv_index_col_name='index', csv_comment_col_name='comment'):
        """
        Shogi-CLIP用のデータセットクラス (コメントをCSVからロード)。

        Args:
            hcpe_file_paths (list or str): .hcpeファイルのパス (単一またはリスト)。
            comment_csv_file_path (str): コメントが格納されたCSVファイルのパス。
            preload_hcpe (bool): Trueの場合、HCPEデータを初期化時にメモリにすべて読み込みます。
            csv_index_col_name (str): CSVファイル内のインデックスが含まれる列の名前。
            csv_comment_col_name (str): CSVファイル内のコメントが含まれる列の名前。
        """
        super().__init__()

        if isinstance(hcpe_file_paths, str):
            hcpe_file_paths = [hcpe_file_paths]
        
        self.hcpe_data = self._load_hcpe_data(hcpe_file_paths, preload_hcpe)
        # コメントをCSVから読み込み、辞書として保持 (インデックス -> コメント文字列)
        self.comments_dict = self._load_comments_from_csv(
            comment_csv_file_path, 
            csv_index_col_name, 
            csv_comment_col_name
        )
        
        if not self.hcpe_data:
            logging.warning("HCPEデータがロードされませんでした。データセットは空になります。")
        if not self.comments_dict:
            logging.warning("コメントデータがロードされませんでした。")

    def _load_hcpe_data(self, file_paths, preload):
        if not preload:
            raise NotImplementedError("遅延ロードは現在実装されていません。preload_hcpe=Trueを使用してください。")

        all_hcpe_records = []
        for path in file_paths:
            if os.path.exists(path):
                logging.info(f"HCPEデータをロード中: {path}")
                try:
                    data = np.fromfile(path, dtype=HuffmanCodedPosAndEvalComment)
                    all_hcpe_records.append(data)
                except Exception as e:
                    logging.error(f"HCPEファイルのロードに失敗しました: {path} - {e}")
            else:
                logging.warning(f"HCPEファイルが見つかりません: {path}")
        
        if not all_hcpe_records:
            return [] # 空のリストを返す
        return np.concatenate(all_hcpe_records)

    def _load_comments_from_csv(self, csv_file_path, index_col_name, comment_col_name):
        comments_map = {}
        if os.path.exists(csv_file_path):
            logging.info(f"コメントデータをCSVからロード中: {csv_file_path}")
            try:
                with open(csv_file_path, 'r', encoding='utf-8', newline='') as csvfile:
                    reader = csv.DictReader(csvfile) # ヘッダー行をキーとして利用
                    if index_col_name not in reader.fieldnames or \
                       comment_col_name not in reader.fieldnames:
                        logging.error(f"CSVファイルに必要な列が見つかりません: '{index_col_name}', '{comment_col_name}'")
                        return {}
                        
                    for row in reader:
                        try:
                            idx = int(row[index_col_name])
                            comment_text = row[comment_col_name]
                            comments_map[idx] = comment_text.strip()
                        except ValueError:
                            logging.warning(f"CSVの行でインデックスを整数に変換できませんでした: {row}")
                        except KeyError:
                            logging.warning(f"CSVの行で指定された列名が見つかりませんでした: {row}")
                logging.info(f"{len(comments_map)} 件のコメントをCSVからロードしました。")
            except Exception as e:
                logging.error(f"コメントCSVファイルのロード/解析に失敗しました: {csv_file_path} - {e}")
        else:
            logging.warning(f"コメントCSVファイルが見つかりません: {csv_file_path}")
        return comments_map

    def __len__(self):
        return len(self.hcpe_data)

    def __getitem__(self, idx):
        if not isinstance(self.hcpe_data, np.ndarray) or idx >= len(self.hcpe_data): # hcpe_dataがリストの場合も考慮
             raise IndexError(f"インデックス {idx} が範囲外です。hcpe_dataの長さ: {len(self.hcpe_data) if isinstance(self.hcpe_data, np.ndarray) else 'N/A (empty list)'}")


        hcpe_record = self.hcpe_data[idx]
        board = Board()
        board.set_hcp(hcpe_record['hcp'])

        features1_sample = np.zeros((FEATURES1_NUM, 9, 9), dtype=np.float32)
        features2_sample = np.zeros((FEATURES2_NUM, 9, 9), dtype=np.float32)
        make_input_features(board, features1_sample, features2_sample)

        comment_idx = hcpe_record['comment_index']
        # 辞書からコメントを取得。存在しない場合はデフォルト値（空文字）を使用。
        raw_comment_text = self.comments_dict.get(comment_idx, "")
        
        if not raw_comment_text: # コメントが見つからなかった場合や元々空の場合
             logging.debug(f"コメントが見つかりません/空です。hcpe_index: {idx}, comment_index_in_hcpe: {comment_idx}")


        return features1_sample, features2_sample, raw_comment_text

# (shogi_clip_collate_fn は変更なしでそのまま利用できます)
def shogi_clip_collate_fn(batch):
    features1_list = [item[0] for item in batch]
    features2_list = [item[1] for item in batch]
    raw_texts_list = [item[2] for item in batch] 

    batched_features1 = torch.from_numpy(np.stack(features1_list))
    batched_features2 = torch.from_numpy(np.stack(features2_list))

    return batched_features1, batched_features2, raw_texts_list

# --- 使用例 ---
# if __name__ == '__main__':
#     # ダミーのHCPEファイルとコメントCSVファイルを作成 (テスト用)
#     DUMMY_HCPE_FILE = "dummy_data.hcpe"
#     DUMMY_COMMENT_CSV_FILE = "dummy_comments.csv"

#     # ダミーHCPEデータの作成
#     sample_hcpe_data = np.zeros(3, dtype=HuffmanCodedPosAndEvalComment)
#     sample_hcpe_data[0]['hcp'] = np.random.randint(0, 256, 32, dtype=np.uint8)
#     sample_hcpe_data[0]['comment_index'] = 101 # CSVのインデックスに対応
#     sample_hcpe_data[1]['hcp'] = np.random.randint(0, 256, 32, dtype=np.uint8)
#     sample_hcpe_data[1]['comment_index'] = 102 # CSVのインデックスに対応
#     sample_hcpe_data[2]['hcp'] = np.random.randint(0, 256, 32, dtype=np.uint8)
#     sample_hcpe_data[2]['comment_index'] = 999 # CSVに存在しないインデックス (テスト用)


#     with open(DUMMY_HCPE_FILE, 'wb') as f:
#         f.write(sample_hcpe_data.tobytes())

#     # ダミーコメントCSVデータの作成 (index,comment のヘッダー付き)
#     dummy_comments_csv_data = [
#         {'index': '101', 'comment': 'これはインデックス101のコメントです。'},
#         {'index': '102', 'comment': ' CSVから読み込んだ2番目の解説。'},
#         {'index': '103', 'comment': '使われないコメント。'} 
#     ]
#     csv_fieldnames = ['index', 'comment']
#     with open(DUMMY_COMMENT_CSV_FILE, 'w', encoding='utf-8', newline='') as f:
#         writer = csv.DictWriter(f, fieldnames=csv_fieldnames)
#         writer.writeheader()
#         writer.writerows(dummy_comments_csv_data)

#     # --- DatasetとDataLoaderのインスタンス化 ---
#     hcpe_files = [DUMMY_HCPE_FILE]
#     comment_csv_file = DUMMY_COMMENT_CSV_FILE
#     batch_size = 2
    
#     # Datasetの作成 (CSVの列名を指定可能)
#     shogi_dataset = ShogiClipDataset(
#         hcpe_file_paths=hcpe_files, 
#         comment_csv_file_path=comment_csv_file,
#         csv_index_col_name='index', # CSVのインデックス列名
#         csv_comment_col_name='comment'  # CSVのコメント列名
#     )

#     shogi_dataloader = DataLoader(
#         shogi_dataset,
#         batch_size=batch_size,
#         shuffle=True,
#         collate_fn=shogi_clip_collate_fn,
#         num_workers=0,
#         pin_memory=True 
#     )

#     if len(shogi_dataset) > 0:
#         logging.info(f"データローダから {len(shogi_dataloader)} バッチを取得します。")
#         for i, (batch_f1, batch_f2, batch_texts) in enumerate(shogi_dataloader):
#             logging.info(f"--- バッチ {i+1} ---")
#             logging.info(f"batch_f1 shape: {batch_f1.shape}")
#             logging.info(f"batch_f2 shape: {batch_f2.shape}")
#             logging.info(f"batch_texts (len: {len(batch_texts)}): {batch_texts}")
#             # hcpe_data[2]のcomment_index=999はCSVにないので、空文字になるはず
#             if any(idx == 999 for idx in sample_hcpe_data[i*batch_size : (i+1)*batch_size]['comment_index']):
#                 assert "" in batch_texts, "存在しないインデックスのコメントが空文字になっていません"
#             if i >= 1: 
#                 break
#     else:
#         logging.warning("データセットが空のため、データローダのテストをスキップします。")

#     if os.path.exists(DUMMY_HCPE_FILE):
#         os.remove(DUMMY_HCPE_FILE)
#     if os.path.exists(DUMMY_COMMENT_CSV_FILE):
#         os.remove(DUMMY_COMMENT_CSV_FILE)

In [ ]:
shogi_dataset = ShogiClipDataset(
    hcpe_file_paths=["path/to/your/data.hcpe"], 
    comment_csv_file_path="path/to/your/comments.csv",
    csv_index_col_name='index_column_name_in_csv', # 例: 'ID' や 'CommentIndex'
    csv_comment_col_name='comment_text_column_name_in_csv' # 例: 'Text' や 'KifuComment'
)